# CHI 2026 Study 2 Analysis
All analysis on study 2 data.

# Imports and setup

In [ ]:
import os
import sys

sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

# IRR for Methods

In [ ]:
import krippendorff

## Data Loading

In [ ]:
# load irr data
# irr_data_df = pd.read_csv(
#     "./coded-data/chi-26-coding-irr_08-26-25.csv", keep_default_na=True
# )
irr_data_df = pd.read_csv(
    "./coded-data/chi-26-coding-irr_08-27-25.csv", keep_default_na=True
)
irr_data_df.head()

## IRR

In [ ]:
mapping = {
    "y": 1,
    "m": 2,
    "n": 3,
}

In [ ]:
# by model
for model in ["Model A", "Model B", "Model C", "Model D"]:
    print(f"{model}")
    reliability_data = []
    for coder in ["Anne Marie", "Xinru", "Dwayne", "Kapil"]:
        curr_ratings = irr_data_df[f"{model} -- {coder}"].to_numpy(na_value=np.nan)
        curr_ratings = [mapping[x] if x in mapping else np.nan for x in curr_ratings]
        reliability_data.append(curr_ratings)
        print(
            f"{coder} -- y: {curr_ratings.count(1)} | m: {curr_ratings.count(2)} | n: {curr_ratings.count(3)} | nan: {curr_ratings.count(np.nan)}"
        )
    print(
        f"Krippendorff's alpha for {model} only: {krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')}"
    )
    # print dashed line
    print("-" * 100)

In [ ]:
# by coder
reliability_data = []
for coder in ["Anne Marie", "Xinru", "Dwayne", "Kapil"]:
    print(f"{coder}")
    ratings_for_coder = []
    for model in ["Model A", "Model B", "Model C", "Model D"]:
        curr_ratings = irr_data_df[f"{model} -- {coder}"].to_numpy(na_value=np.nan)
        curr_ratings = [mapping[x] if x in mapping else np.nan for x in curr_ratings]
        ratings_for_coder.extend(curr_ratings)

    # print y, n, m, and nan
    print(
        f"y: {ratings_for_coder.count(1)} | m: {ratings_for_coder.count(2)} | n: {ratings_for_coder.count(3)} | nan: {ratings_for_coder.count(np.nan)}"
    )
    reliability_data.append(ratings_for_coder)
    print("-" * 100)

print(
    f"Krippendorff's alpha across all models: {krippendorff.alpha(reliability_data=reliability_data, level_of_measurement='nominal')}"
)

# Numbers for Methods
Compute summary stats about the dataset for the methods.

In [ ]:
from scripts.data_loader import generate_target_dataset

In [ ]:
dataset_to_caption = generate_target_dataset(
    "../data/caption-dataset/annotations/train.json",
    "../data/image-quality-assessment/annotations/train.json",
)
dataset_to_caption_df = pd.DataFrame.from_dict(dataset_to_caption)
dataset_to_caption_df.head()

In [ ]:
# filter by captionability
filtered_cap_data_df = dataset_to_caption_df[
    (dataset_to_caption_df["unrecognizable"] <= 2)
    & (dataset_to_caption_df["text_detected"])
]
print(
    f"Filtered dataset of {len(filtered_cap_data_df)} ({len(filtered_cap_data_df) / len(dataset_to_caption_df) * 100:.2f}%) images"
)
# Main analysis

In [ ]:
threshold = 1
high_quality_filtered_df = filtered_cap_data_df[
    (filtered_cap_data_df["framing"] <= threshold)
    & (filtered_cap_data_df["rotation"] <= threshold)
    & (filtered_cap_data_df["blur"] <= threshold)
    & (filtered_cap_data_df["obstruction"] <= threshold)
    & (filtered_cap_data_df["too dark"] <= threshold)
    & (filtered_cap_data_df["too bright"] <= threshold)
    & (filtered_cap_data_df["other"] <= threshold)
    & (filtered_cap_data_df["no issue"] >= 4)
]
display(high_quality_filtered_df.describe())
print(
    f"Number of high quality images: {len(high_quality_filtered_df)} ({len(high_quality_filtered_df) / len(dataset_to_caption_df) * 100:.1f}%)"
)

In [ ]:
threshold = 4
low_quality_filtered_df = filtered_cap_data_df[
    (filtered_cap_data_df["framing"] >= threshold)
    | (filtered_cap_data_df["rotation"] >= threshold)
    | (filtered_cap_data_df["blur"] >= threshold)
    | (filtered_cap_data_df["obstruction"] >= threshold)
    | (filtered_cap_data_df["too dark"] >= threshold)
    | (filtered_cap_data_df["too bright"] >= threshold)
    | (filtered_cap_data_df["other"] >= threshold)
]
display(low_quality_filtered_df.describe())
print(
    f"Number of low quality images: {len(low_quality_filtered_df)} ({len(low_quality_filtered_df) / len(dataset_to_caption_df) * 100:.1f}%)"
)

# Main analysis
Now we'll do the main analysis for the paper

## Helper functions

### Quality counting

In [ ]:
def calculate_quality_metrics(df, reference_df=None, quality_columns=None):
    """
    Calculate quality issue counts and optionally percentages compared to a reference dataset.

    Args:
        df: DataFrame containing the quality issues data
        reference_df: Optional reference DataFrame to calculate percentages against
        quality_columns: List of quality issue column names. If None, uses default columns

    Returns:
        DataFrame with quality counts, and optionally percentages if reference_df is provided
    """
    if quality_columns is None:
        quality_columns = [
            "unrecognizable",
            "blur",
            "framing",
            "obstruction",
            "rotation",
            "too dark",
            "too bright",
            "other",
        ]

    # Calculate counts
    quality_counts = pd.concat(
        [df[col].value_counts() for col in quality_columns], axis=1
    )
    quality_counts.columns = quality_columns

    # Replace NaN with 0 and convert to int
    quality_counts = quality_counts.fillna(0).astype(int)

    # sort index from 0 to 5
    quality_counts = quality_counts.sort_index()

    # Add total row
    quality_counts.loc["total"] = quality_counts.sum()

    # Calculate percentages if reference DataFrame is provided
    if reference_df is not None:
        reference_counts = calculate_quality_metrics(
            reference_df, quality_columns=quality_columns
        )
        quality_percentages = quality_counts.div(reference_counts, axis=0) * 100
        return quality_percentages.round(2)

    return quality_counts


def combine_counts_and_percentages(counts_df, percentages_df=None):
    """
    Combines counts and percentages into a single DataFrame with formatted strings.

    Args:
        counts_df: DataFrame containing the counts
        percentages_df: Optional DataFrame containing percentages. If None, percentages
                       will be calculated using the total row of counts_df

    Returns:
        DataFrame with formatted strings combining counts and percentages
    """
    # Calculate percentages if not provided
    if percentages_df is None:
        percentages_df = (counts_df.div(counts_df.loc["total"], axis=1) * 100).round(2)

    def format_count_and_percentage(count, percentage):
        count_str = (
            str(int(float(count))) if float(count).is_integer() else str(float(count))
        )
        return f"{count_str} ({percentage:.2f}%)"

    # Create combined DataFrame
    combined_stats = pd.DataFrame(
        [
            [
                format_count_and_percentage(count, pct)
                for count, pct in zip(row_counts, row_pcts)
            ]
            for row_counts, row_pcts in zip(counts_df.values, percentages_df.values)
        ],
        index=counts_df.index,
        columns=counts_df.columns,
    )

    return combined_stats

### Accuracy tables

In [ ]:
# get only the images that were verifable
def get_accuracy_counts(dataframe_with_counts, model_names):
    accuracy_counts_df = pd.concat(
        [
            dataframe_with_counts[f"{model}_correct"]
            .replace(True, "True")
            .replace(False, "False")
            .value_counts()
            for model in model_names
        ],
        axis=1,
    )

    accuracy_counts_df.columns = model_names

    # replace nan with 0
    accuracy_counts_df = accuracy_counts_df.fillna(0)

    # add a percentage to each count cell
    accuracy_counts_df = accuracy_counts_df.apply(
        lambda x: x.apply(
            lambda y: f"{y} ({round(100 * y / len(dataframe_with_counts), 1):.1f}%)"
        )
    )

    # display(accuracy_counts_df)
    # display(round(100 * accuracy_counts_df / len(dataframe_with_counts), 1))

    return accuracy_counts_df


def _compute_accuracy_counts(dataframe, model_columns):
    """Compute accuracy counts for given model columns.

    Args:
        dataframe (pandas.DataFrame): Input dataframe
        model_columns (list): List of model column names

    Returns:
        pandas.DataFrame: DataFrame with accuracy counts for each model
    """
    accuracy_counts = pd.concat(
        [
            dataframe[f"{model}_correct"]
            .replace(True, "True")
            .replace(False, "False")
            .value_counts()
            for model in model_columns
        ],
        axis=1,
    )
    accuracy_counts.columns = model_columns
    return accuracy_counts


def _create_combined_counts_df(accuracy_counts_df, total_count):
    """Create a combined dataframe with counts and percentages.

    Args:
        accuracy_counts_df (pandas.DataFrame): DataFrame with accuracy counts
        total_count (int): Total number of samples

    Returns:
        pandas.DataFrame: Combined dataframe with counts and percentages
    """
    accuracy_pct_df = round(100 * accuracy_counts_df / total_count, 2)

    combined_df = pd.DataFrame(
        index=accuracy_counts_df.index, columns=accuracy_counts_df.columns
    )

    for col in accuracy_counts_df.columns:
        combined_df[col] = (
            accuracy_counts_df[col].apply(lambda x: f"{x:.0f}")
            + " "
            + accuracy_pct_df[col].apply(lambda x: f"({round(x, 1):.1f}%)")
        )

    return combined_df


def create_accuracy_table(
    dataframe_with_counts,
    quality_columns,
    model_columns,
    include_overall=True,
    include_incorrect=False,
):
    """Create a table of accuracy for each image quality issue

    Args:
        dataframe_with_counts (pandas dataframe): table that includes image quality issues and columns for each model, with values of yes, yes++, or no.
        quality_columns (list): list of image quality issues to include in the table
        include_overall (bool, optional): whether to include an overall accuracy row. Defaults to True.
        include_incorrect (bool, optional): whether to include a row for the number of incorrect predictions. Defaults to False.

    Returns:
        pandas dataframe: table of accuracy for each image quality issue. Table will have 3 columns (one for each model) and 1-2 rows for each image quality issue (depending if include incorrect is true). If include_overall, then first row will include overall accuracy.
    """
    output_df = pd.DataFrame()
    total_count = len(dataframe_with_counts)
    to_include = ["True", "False"] if include_incorrect else ["True"]

    # Add overall row if requested
    if include_overall:
        accuracy_counts_df = _compute_accuracy_counts(
            dataframe_with_counts, model_columns
        )
        combined_df = _create_combined_counts_df(accuracy_counts_df, total_count)
        overall_text = "Overall"

        # add a column named Num. Images that includes count (percentage)
        combined_df["Num. Images"] = (
            f"{len(dataframe_with_counts)} ({round(100 * len(dataframe_with_counts) / total_count, 1):.1f}%)"
        )

        overall_row_df = combined_df.loc[to_include]
        overall_row_df["issue"] = overall_text
        output_df = pd.concat([output_df, overall_row_df], axis=0)

    # Process each quality issue
    for issue, issue_text in quality_columns.items():
        relevant_df = dataframe_with_counts[dataframe_with_counts[issue]]
        if len(relevant_df) == 0:
            continue

        # compute accuracy counts
        accuracy_counts_df = _compute_accuracy_counts(relevant_df, model_columns)
        accuracy_counts_df = accuracy_counts_df.fillna(0)
        combined_df = _create_combined_counts_df(accuracy_counts_df, len(relevant_df))

        # add a column named Num. Images that includes count (percentage)
        combined_df["Num. Images"] = (
            f"{len(relevant_df)} ({round(100 * len(relevant_df) / total_count, 1):.1f}%)"
        )

        issue_row_df = combined_df.loc[to_include]
        issue_row_df["issue"] = issue_text
        output_df = pd.concat([output_df, issue_row_df], axis=0)

    # reset index
    output_df.reset_index(drop=False, inplace=True)
    output_df.rename(columns={"index": "Correct?"}, inplace=True)

    # move issue and index column to the front
    output_df = output_df[
        ["issue", "Correct?", "Num. Images"]
        + [
            col
            for col in output_df.columns
            if col != "issue" and col != "Correct?" and col != "Num. Images"
        ]
    ]

    return output_df

## Constants

In [ ]:
ANON_TO_MODEL = {
    "Model A": "Llama-3.2-90B-Vision-Instruct-bnb-4bit",
    "Model B": "gemini-2.5-flash",
    "Model C": "gpt-4.1-2025-04-14",
    "Model D": "Molmo-72B-0924-nf4",
}

MODEL_SHORT_NAMES = {
    "gpt-4.1-2025-04-14": "gpt-4.1",
    "gemini-2.5-flash": "gemini-2.5-flash",
    "Llama-3.2-90B-Vision-Instruct-bnb-4bit": "llama-90b-4bit",
    "Molmo-72B-0924-nf4": "molmo-72b-4bit",
}

MODEL_MARKED_COLS = {
    "Model A Marked (Llama)": "llama-90b-4bit_marked",
    "Model B Marked (Gemini)": "gemini-2.5-flash_marked",
    "Model C Marked (GPT)": "gpt-4.1_marked",
    "Model D Marked (Molmo)": "molmo-72b-4bit_marked",
}

## Data Loading

In [ ]:
target_images_dtypes = {
    "id": str,
    "orig_id": str,
    "file_name": str,
    "image_url": str,
    "image_preview": str,
    "type": str,
    "human_captions": str,
    "expert_caption": str,
    "orig annotator": str,
    "orig annotation notes": str,
    "unable_to_verify": str,
    "double code notes": str,
    "double verified": str,
    "annotator": str,
    "annotation notes": str,
    "object": str,
    "product": str,
    "brand": str,
    "variety": str,
    "double annotator": str,
    "double annotation": str,
    "coder": str,
    "coding_notes": str,
    "double_coder": str,
    "double_coding_notes": str,
    "gpt-4.1": str,
    "gemini-2.5-flash": str,
    "llama-90B-4bit": str,
    "molmo-72B-4bit": str,
    "Model A Marked": str,
    "Model A Correct?": str,
    "Model B Marked": str,
    "Model B Correct?": str,
    "Model C Marked": str,
    "Model C Correct?": str,
    "Model D Marked": str,
    "Model D Correct?": str,
    "text_detected": str,
    "format annotator": str,
    "format annotator double coder": str,
    "curved label": str,
    "text panel": str,
    "unrecognizable": bool,
    "framing": bool,
    "blur": bool,
    "obstruction": bool,
    "rotation": bool,
    "too dark": bool,
    "too bright": bool,
    "other": bool,
    "unrecognizable_orig": int,
    "framing_orig": int,
    "blur_orig": int,
    "obstruction_orig": int,
    "rotation_orig": int,
    "too_dark_orig": int,
    "too_bright_orig": int,
    "other_orig": int,
    "no_issue_orig": int,
    "Notes": str,
    "Should remove?": str,
}

In [ ]:
annotated_df = pd.read_csv(
    # "./coded-data/chi-26-coded-final_09-05-25.csv",
    "./coded-data/chi-26-coded-final_09-23-25.csv",
    dtype=target_images_dtypes,
    keep_default_na=False,
)

# filter out any Should remove? == "Yes"
annotated_df = annotated_df[annotated_df["Should remove?"] != "YES"]
print(f"Total number of images: {len(annotated_df)}")
annotated_df.head()

In [ ]:
annotated_df[
    (annotated_df["type"] == "high-quality")
    & (annotated_df["unrecognizable_orig"] == 2)
].to_dict(orient="records")

In [ ]:
# Prepare columns
orig_cols = [col for col in annotated_df.columns if col.endswith("_orig")]
types = sorted(annotated_df["type"].unique())
num_captioners_range = range(0, 6)

# Prepare a list to collect rows for the final table
table_rows = []

for typ in types:
    for n in num_captioners_range:
        row = {
            "Image Type": typ,
            "Num. Captioners": n,
        }
        for col in orig_cols:
            col_name = col.replace("_orig", "").replace("_", " ")
            col_name = col_name.title()
            mask = (annotated_df[col] == n) & (annotated_df["type"] == typ)
            count = mask.sum()
            total = (annotated_df["type"] == typ).sum()
            percent = (count / total * 100) if total > 0 else 0
            row[col_name] = f"{count} ({percent:.2f}%)"
        table_rows.append(row)

# Convert to DataFrame
table_df = pd.DataFrame(table_rows)
table_df

In [ ]:
# clean-up table
cols_to_drop = [
    # misc
    "image_preview",
    # old annotations
    "human_captions",
    "expert_caption",
    "orig annotator",
    "orig annotation notes",
    "unable_to_verify",
    "double code notes",
    "double verified",
    # old image quality
    "unrecognizable_orig",
    "framing_orig",
    "blur_orig",
    "obstruction_orig",
    "rotation_orig",
    "too_dark_orig",
    "too_bright_orig",
    "other_orig",
    "no_issue_orig",
    "Notes",
    "Should remove?",
]
annotated_df = annotated_df.drop(columns=cols_to_drop)

# rename Model A-D Marked and Correct? with the short model name
for model in ["Model A", "Model B", "Model C", "Model D"]:
    # replace y-n with true and false
    annotated_df[f"{model} Correct?"] = annotated_df[f"{model} Correct?"].map(
        {"y": True, "n": False}
    )
    annotated_df = annotated_df.rename(
        columns={
            # f"{model} Marked": f"{MODEL_SHORT_NAMES[ANON_TO_MODEL[model]]}_marked",
            f"{model} Correct?": f"{MODEL_SHORT_NAMES[ANON_TO_MODEL[model]]}_correct",
        }
    )
    annotated_df = annotated_df.rename(columns=MODEL_MARKED_COLS)
    display(
        annotated_df[
            f"{MODEL_SHORT_NAMES[ANON_TO_MODEL[model]]}_correct"
        ].value_counts()
    )

# fill blank curved label and text panel with false
annotated_df["curved label"] = annotated_df["curved label"].map(
    {"TRUE": True, "": False}
)
annotated_df["text panel"] = annotated_df["text panel"].map({"TRUE": True, "": False})

# overall summary stats
print(f"Total dataset size: {len(annotated_df)}")
display(annotated_df["type"].value_counts())
annotated_df.head()

In [ ]:
# misc cleaning
annotated_df.drop(
    columns=[
        "Column 62",
        "Num models correct",
        "annotator",
        "annotation notes",
        "double annotator",
        "double annotation",
        "coder",
        "coding_notes",
        "double_coder",
        "double_coding_notes",
        "format annotator",
        "format annotator double coder",
        "llama-90b-4bit_marked",
        "gemini-2.5-flash_marked",
        "gpt-4.1_marked",
        "molmo-72b-4bit_marked",
        "format annotator",
        "format annotator double coder",
    ],
    inplace=True,
)
annotated_df.head()

In [ ]:
# save cleaned df
os.makedirs("./coded-data/cleaned", exist_ok=True)
annotated_df.to_csv("./coded-data/cleaned/chi-26-coded-final_09-23-25.csv", index=False)

### Export data for sampling

In [ ]:
# create a sample of all matched images, and n high and low quality images
# used for sampling data
n_samples = 400
sample_df = pd.concat(
    [
        annotated_df[annotated_df["type"] == "matched-image"],
        annotated_df[annotated_df["type"] == "high-quality"][:n_samples],
        annotated_df[annotated_df["type"] == "low-quality"][:n_samples],
    ]
)
sample_df.to_csv(
    f"./coded-data/cleaned/final-image-sample_{len(sample_df)}-images_09-23-25.csv",
    index=False,
)

# also create a remaining sample of all images of annotated_df that are not in the sample_df
remaining_df = annotated_df[~annotated_df["id"].isin(sample_df["id"])]
remaining_df.to_csv(
    f"./coded-data/cleaned/remaining-images_{len(remaining_df)}-images_09-23-25.csv",
    index=False,
)

### Create group codes

In [ ]:
# no issues
annotated_df["no_issues"] = (
    (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)

# single issues
annotated_df["blur_only"] = (
    (annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["framing_only"] = (
    (annotated_df["framing"])
    & (~annotated_df["blur"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["rotation_only"] = (
    (annotated_df["rotation"])
    & (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["obstruction_only"] = (
    (annotated_df["obstruction"])
    & (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["too_dark_only"] = (
    (annotated_df["too dark"])
    & (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["too_bright_only"] = (
    (annotated_df["too bright"])
    & (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["other"])
)
annotated_df["other_only"] = (
    (annotated_df["other"])
    & (~annotated_df["blur"])
    & (~annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
)

# cooccurring issues
annotated_df["blur_framing"] = (
    (annotated_df["blur"])
    & (annotated_df["framing"])
    & (~annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["blur_rotation"] = (
    (annotated_df["blur"])
    & (annotated_df["rotation"])
    & (~annotated_df["framing"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["framing_rotation"] = (
    (annotated_df["framing"])
    & (annotated_df["rotation"])
    & (~annotated_df["blur"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)
annotated_df["blur_framing_rotation"] = (
    (annotated_df["blur"])
    & (annotated_df["framing"])
    & (annotated_df["rotation"])
    & (~annotated_df["obstruction"])
    & (~annotated_df["too dark"])
    & (~annotated_df["too bright"])
    & (~annotated_df["other"])
)

# other co-occurring issues are cases where there are multiple issues but not the co-occurring issues above
annotated_df["multiple_issues"] = (
    annotated_df[
        [
            "blur",
            "framing",
            "rotation",
            "obstruction",
            "too dark",
            "too bright",
            "other",
        ]
    ].sum(axis=1)
    >= 2
)

annotated_df["not_interesting_cooccurring"] = ~(
    (annotated_df["blur_framing"])
    | (annotated_df["blur_rotation"])
    | (annotated_df["framing_rotation"])
    | (annotated_df["blur_framing_rotation"])
)

annotated_df["other_cooccurring"] = (
    annotated_df["multiple_issues"] & annotated_df["not_interesting_cooccurring"]
)


combine_counts_and_percentages(
    calculate_quality_metrics(
        annotated_df,
        quality_columns=[
            # no issues
            "no_issues",
            # individual issues
            "blur_only",
            "framing_only",
            "rotation_only",
            "obstruction_only",
            "too_dark_only",
            "too_bright_only",
            "other_only",
            # cooccurring issues
            "blur_framing",
            "blur_rotation",
            "framing_rotation",
            "blur_framing_rotation",
            "other_cooccurring",
        ],
    ),
)

## Analysis 1: how accurately do VLMs identify products, given image quality issues?

### Overall accuracy

In [ ]:
for type in ["matched-image", "high-quality", "low-quality"]:
    print(f"Type: {type}")
    display(
        get_accuracy_counts(
            annotated_df[annotated_df["type"] == type], MODEL_SHORT_NAMES.values()
        )
    )

### By quality issue

In [ ]:
quality_issues = {
    # no issues
    "no_issues": "no issues",
    # individual issues
    "blur_only": "blur only",
    "framing_only": "framing only",
    "rotation_only": "rotation only",
    "obstruction_only": "obstruction only",
    "too_dark_only": "too dark only",
    "too_bright_only": "too bright only",
    "other_only": "other only",
    # cooccurring issues
    "blur_framing": "blur and framing",
    "blur_rotation": "blur and rotation",
    "framing_rotation": "framing and rotation",
    "blur_framing_rotation": "blur, framing, and rotation",
    "other_cooccurring": "other cooccurring",
}

In [ ]:
create_accuracy_table(
    annotated_df,
    quality_issues,
    MODEL_SHORT_NAMES.values(),
    include_overall=True,
    include_incorrect=False,
)

In [ ]:
create_accuracy_table(
    annotated_df[annotated_df["type"] == "matched-image"],
    quality_issues,
    MODEL_SHORT_NAMES.values(),
    include_overall=True,
    include_incorrect=False,
)

In [ ]:
create_accuracy_table(
    annotated_df[annotated_df["type"] == "high-quality"],
    quality_issues,
    MODEL_SHORT_NAMES.values(),
    include_overall=True,
    include_incorrect=False,
)

In [ ]:
create_accuracy_table(
    annotated_df[annotated_df["type"] == "low-quality"],
    quality_issues,
    MODEL_SHORT_NAMES.values(),
    include_overall=True,
    include_incorrect=False,
)

## Analysis 2: How do curved labels and text panels affect accuracy?

In [ ]:
# let's also look at rounded and text panel images only
annotated_df_curved = annotated_df[
    (annotated_df["curved label"]) & ~annotated_df["text panel"]
]
annotated_df_text = annotated_df[
    (annotated_df["text panel"]) & ~annotated_df["curved label"]
]
annotated_df_curved_text = annotated_df[
    (annotated_df["curved label"]) & (annotated_df["text panel"])
]
annotated_df_not_curved_no_text = annotated_df[
    (~annotated_df["curved label"]) & (~annotated_df["text panel"])
]

print(
    f"Number of images with curved label: {len(annotated_df_curved)} ({len(annotated_df_curved) / len(annotated_df) * 100:.2f}%)"
)
print(
    f"Number of images with text panel: {len(annotated_df_text)} ({len(annotated_df_text) / len(annotated_df) * 100:.2f}%)"
)
print(
    f"Number of images with curved label and text panel: {len(annotated_df_curved_text)} ({len(annotated_df_curved_text) / len(annotated_df) * 100:.2f}%)"
)
print(
    f"Number of images without curved label and text panel: {len(annotated_df_not_curved_no_text)} ({len(annotated_df_not_curved_no_text) / len(annotated_df) * 100:.2f}%)"
)

In [ ]:
def compute_count_pct_str(subset_df, total_df):
    return f"{len(subset_df)} ({100 * len(subset_df) / len(total_df):.2f}%)"


image_types = ["matched-image", "high-quality", "low-quality"]
product_attributes = ["curved only", "text panel only", "curved and text panel"]

for type in image_types:
    print(f"Type: {type}")
    print(
        f"Total number of images: {len(annotated_df[annotated_df['type'] == type])}",
        end="\n\n",
    )

    # baseline accuracy
    print(
        f"Baseline accuracy: {compute_count_pct_str(annotated_df[annotated_df['type'] == type], annotated_df[annotated_df['type'] == type])}"
    )
    display(
        get_accuracy_counts(
            annotated_df[annotated_df["type"] == type], MODEL_SHORT_NAMES.values()
        )
    )

    # accuracy for images without any image attributes
    print(
        f"Accuracy for images without any image attributes: {compute_count_pct_str(annotated_df_not_curved_no_text[annotated_df_not_curved_no_text['type'] == type], annotated_df[annotated_df['type'] == type])}"
    )
    display(
        get_accuracy_counts(
            annotated_df_not_curved_no_text[
                annotated_df_not_curved_no_text["type"] == type
            ],
            MODEL_SHORT_NAMES.values(),
        )
    )

    for image_attr in product_attributes:
        print(f"Image attribute: {image_attr}", end=" -- ")
        if image_attr == "curved only":
            print(
                f"{compute_count_pct_str(annotated_df_curved[annotated_df_curved['type'] == type], annotated_df[annotated_df['type'] == type])}"
            )
            display(
                get_accuracy_counts(
                    annotated_df_curved[annotated_df_curved["type"] == type],
                    MODEL_SHORT_NAMES.values(),
                )
            )
        elif image_attr == "text panel only":
            print(
                f"{compute_count_pct_str(annotated_df_text[annotated_df_text['type'] == type], annotated_df[annotated_df['type'] == type])}"
            )
            display(
                get_accuracy_counts(
                    annotated_df_text[annotated_df_text["type"] == type],
                    MODEL_SHORT_NAMES.values(),
                )
            )
        elif image_attr == "curved and text panel":
            print(
                f"{compute_count_pct_str(annotated_df_curved_text[annotated_df_curved_text['type'] == type], annotated_df[annotated_df['type'] == type])}"
            )
            display(
                get_accuracy_counts(
                    annotated_df_curved_text[annotated_df_curved_text["type"] == type],
                    MODEL_SHORT_NAMES.values(),
                )
            )
    print("-" * 100)

## Analysis 3: Regression of Image Quality and Correctness

In [ ]:
columns_to_include = [
    "id",
    "file_name",
    "image_url",
    "type",
    "curved label",
    "text panel",
    "blur",
    "framing",
    "rotation",
    "blur_framing",
    "blur_rotation",
    "framing_rotation",
    "blur_framing_rotation",
    *[f"{x}_correct" for x in MODEL_SHORT_NAMES.values()],
]

# filtered dataframe
regression_df = annotated_df[columns_to_include]

# include only high-quality images and low-quality images
regression_df = regression_df[
    regression_df["type"].isin(["high-quality", "low-quality"])
]

# convert true / false to 1 / 0
regression_df.replace({True: 1, False: 0}, inplace=True)

# display before printing
print(f"Number of images: {len(regression_df)}")
display(regression_df["type"].value_counts())
display(regression_df.head())


os.makedirs("./regression-data", exist_ok=True)
regression_df.to_csv(
    f"./regression-data/regression-final-df_{len(regression_df)}-images.csv",
    index=False,
)